# Stance Classifier

Experimental notebook for developing, training, and testing the classification model for the catcher stance.

In [50]:
import pandas as pd
import numpy as np
import os
import ast
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [55]:
# Load the dataset
DATASET_PATH = "./drive/MyDrive/keypoints-labeled-INCOMPLETE.csv"
df = pd.read_csv(DATASET_PATH)

# Convert stringified list -> numpy array
df["features"] = df["features"].apply(ast.literal_eval)
df["features"] = df["features"].apply(lambda x: np.array(x, dtype=np.float32))

# Stack into X matrix
X = np.stack(df["features"].values)

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["choice"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", label_encoder.classes_)

X shape: (429, 238)
y shape: (429,)
Classes: ['Knee-down Right']


In [56]:
# Create the train, test, and validation sets

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (300, 238) (300,)
Val: (64, 238) (64,)
Test: (65, 238) (65,)


In [57]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
LR = 1e-3
EPOCHS = 30
MODEL_OUTPUT_PATH = "catcher_stance/models/catcher_stance_mlp.pt"
LABEL_ENCODER_PATH = "catcher_stance/models/label_encoder.pkl"

import os
os.makedirs("catcher_stance/models", exist_ok=True)

In [58]:
class CatcherDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [59]:
train_dataset = CatcherDataset(X_train, y_train)
val_dataset = CatcherDataset(X_val, y_val)
test_dataset = CatcherDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [60]:
class CatcherMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [61]:
input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))

model = CatcherMLP(input_dim=input_dim, num_classes=num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [62]:
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(train_mode):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

In [63]:
best_val_acc = -1
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_OUTPUT_PATH)

Epoch 1/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 2/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 3/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 4/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 5/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 6/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 7/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 8/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 9/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 10/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 11/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 12/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 13/30 | train_loss=

In [64]:
# Load best model before final test evaluation
best_model = CatcherMLP(input_dim=input_dim, num_classes=num_classes).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_OUTPUT_PATH, map_location=DEVICE))
best_model.eval()

CatcherMLP(
  (net): Sequential(
    (0): Linear(in_features=238, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [65]:
def get_predictions(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    return np.array(all_labels), np.array(all_preds)

In [66]:
y_true, y_pred = get_predictions(best_model, test_loader)

test_acc = accuracy_score(y_true, y_pred)
print("Test Accuracy:", test_acc)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Test Accuracy: 1.0

Classification Report:
                 precision    recall  f1-score   support

Knee-down Right       1.00      1.00      1.00        65

       accuracy                           1.00        65
      macro avg       1.00      1.00      1.00        65
   weighted avg       1.00      1.00      1.00        65


Confusion Matrix:
[[65]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [67]:
# Save label encoder too
joblib.dump(label_encoder, LABEL_ENCODER_PATH)

# Optional: save metrics/history
joblib.dump(history, "catcher_stance/models/training_history.pkl")

print("Saved model to:", MODEL_OUTPUT_PATH)
print("Saved label encoder to:", LABEL_ENCODER_PATH)

Saved model to: catcher_stance/models/catcher_stance_mlp.pt
Saved label encoder to: catcher_stance/models/label_encoder.pkl
